<a href="https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Description:For the Ranking Signal Analysis lane, we target pages that already achieve strong visibility (average search position $\le 10$) and substantial impression volume ($> 500$ impressions), but suffer from below-expected Click-Through Rates (CTR $< 3\%$). These pages represent high-leverage opportunities where search engine users are seeing the result, but not clicking.Signal 1 Verdict (Flag-linked — CTR vs. Position): CONFIRMEDReasoning: Pages sitting in top positions with low CTR consistently correlate with titles/snippets that fail to match user search intent.Signal 2 Verdict (Impression Volume vs. Position): CONFIRMEDReasoning: High impression counts in top positions indicate active search demand, ensuring that CTR improvements translate directly into traffic gains.Reason Codes:LOW_CTR_HIGH_VISIBILITY: Search position $\le 10$ with CTR $< 3\%$ and impressions $> 500$.PERFORMING_AS_EXPECTED: Default status for pages meeting benchmark expectations.Action Label: OPTIMIZE_METADATA_AND_SNIPPET

In [ ]:
import pandas as pd
import numpy as np
import os

# File path check & automatic fallback download for Colab
file_path = '../data/raw/content_refresh_anonymized.csv'

if not os.path.exists(file_path):
    # Try alternative local paths or download directly from starter repo
    if os.path.exists('data/raw/content_refresh_anonymized.csv'):
        file_path = 'data/raw/content_refresh_anonymized.csv'
    else:
        print("Local file not found, downloading anonymized dataset...")
        os.makedirs('../data/raw', exist_ok=True)
        url = 'https://raw.githubusercontent.com/ggirlrottingg/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
        df = pd.read_csv(url)
        df.to_csv(file_path, index=False)

df = pd.read_csv(file_path)
print(f"Successfully loaded dataset with {len(df)} rows.")

Successfully loaded dataset with 30000 rows.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Ensure output directory exists
os.makedirs('../outputs', exist_ok=True)

# 2. Load dataset safely
file_path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(file_path):
    if os.path.exists('data/raw/content_refresh_anonymized.csv'):
        file_path = 'data/raw/content_refresh_anonymized.csv'
    else:
        url = 'https://raw.githubusercontent.com/ggirlrottingg/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
        df = pd.read_csv(url)
        os.makedirs('../data/raw', exist_ok=True)
        df.to_csv(file_path, index=False)

df = pd.read_csv(file_path)

# 3. Use actual columns present in content_refresh_anonymized.csv
imp_col = 'impressions_90d' if 'impressions_90d' in df.columns else 'impressions_last_30d'
pos_col = 'avg_position' if 'avg_position' in df.columns else 'average_position'
ctr_col = 'ctr'

# 4. Benchmark expected CTR based on average position
expected_ctr = np.where(df[pos_col] <= 3, 0.15, np.where(df[pos_col] <= 10, 0.05, 0.01))

# 5. Calculate Action Score
df['ctr_gap'] = np.maximum(0, expected_ctr - df[ctr_col])
df['action_score'] = df[imp_col] * df['ctr_gap']

# 6. Assign Reason Code and Action Label
df['reason_code'] = np.where(
    (df[pos_col] <= 10) & (df[ctr_col] < 0.03) & (df[imp_col] > 500),
    'LOW_CTR_HIGH_VISIBILITY',
    'PERFORMING_AS_EXPECTED'
)

df['action_label'] = np.where(
    df['reason_code'] == 'LOW_CTR_HIGH_VISIBILITY',
    'OPTIMIZE_METADATA_AND_SNIPPET',
    'MONITOR'
)

# 7. Sort and export queue to work/outputs/baseline_action_score.csv
ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)
output_path = '../outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f"Successfully exported {len(ranked_queue)} rows to {output_path}")

Successfully exported 30000 rows to ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Select key columns present in ranked_queue to display top 10 items
display_cols = ['content_id', 'impressions_90d', 'ctr', 'avg_position', 'action_score', 'reason_code', 'action_label']
available_cols = [c for c in display_cols if c in ranked_queue.columns]

ranked_queue[available_cols].head(10)

,content_id,impressions_90d,ctr,avg_position,action_score,reason_code,action_label
0,content_8451fc6f034d,272144,0.03,2.3,32657.28,PERFORMING_AS_EXPECTED,MONITOR
1,content_4a6607efcb46,128068,0.01,2.2,17929.52,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
2,content_e12868d1f396,149712,0.07,2.9,11976.96,PERFORMING_AS_EXPECTED,MONITOR
3,content_c8e9d6ab9013,208678,0.00,9.7,10433.90,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
4,content_453722754fea,140079,0.01,7.6,5603.16,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
5,content_39881853ef0c,112434,0.01,7.2,4497.36,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
6,content_c84a0ab98e90,223271,0.03,7.8,4465.42,PERFORMING_AS_EXPECTED,MONITOR
7,content_8053a66bd6ac,52687,0.08,2.6,3688.09,PERFORMING_AS_EXPECTED,MONITOR
8,content_0919dd345d80,119217,0.02,7.0,3576.51,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
9,content_c1fe78bc4e37,134055,0.03,7.5,2681.10,PERFORMING_AS_EXPECTED,MONITOR


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Identified:

**Informational Query False Positives:** Pages ranking for broad definition terms get high impressions, but naturally low CTR because searchers satisfy their intent on the SERP itself.

**Low-Intent Impressions:** Certain pages capture accidental impressions on secondary keywords, artificially inflating the impression count and skewing the action_score.

Leakage Confirmation:

**No Future Data:** All calculated features (ctr, impressions_90d, avg_position) are derived solely from historical performance windows.

**No Label or Product Flags Used:** The heuristic relies strictly on raw input attributes without incorporating internal product labels or downstream target indicators.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.